In [32]:
p_bronze_account = "saedmsprdizadls01"
p_bronze_container_src = "caas-edms-bronze/air transport/ati/raw/"
p_bronze_container_dst = "caas-edms-bronze/air transport/ati/raw/"
p_bronze_container_arch = "caas-edms-bronze/air transport/ati/raw/archive"
fmt = "%d %b %y"
secret_name = "scr-edms-decrypt-pwd"
p_worksheet = "PAX_BREAKDOWN|FREIGHTER_BREAKDOWN|Airline List|City Links"
p_original_source_name = "ATI"
p_date_normalization_enabled = True


In [33]:
import os
import io
import re
import pandas as pd
import openpyxl
from notebookutils import mssparkutils 
from datetime import datetime
from notebookutils import mssparkutils
from azure.storage.blob import BlobServiceClient, ContainerClient

In [34]:
# Secrets and configurations
password = bytes(mssparkutils.credentials.getSecret("kv-edms-prdiz-001", secret_name, "ls_kv_edms_prd01"), 'ascii')
accountkey = mssparkutils.credentials.getSecret("kv-edms-prdiz-001", "scr-edms-saedmsprdizadls01-key", "ls_kv_edms_prd01")
conn_str = f"DefaultEndpointsProtocol=https;AccountName={p_bronze_account};AccountKey={accountkey};EndpointSuffix=core.windows.net"

In [35]:
# Initialize Blob Service Client
blob_service_client = BlobServiceClient.from_connection_string(conn_str)

# Split container and paths
p_bronze_container = p_bronze_container_src.split("/", 1)[0]
p_bronze_container_src_path = p_bronze_container_src.split("/", 1)[-1]

processed_folder_path = p_bronze_container_src_path.replace("raw", "processed")
p_bronze_container_arch = p_bronze_container_src_path.replace("raw", "archive")

# Initialize container clients
src_container_client = blob_service_client.get_container_client(p_bronze_container)
des_container_client = blob_service_client.get_container_client(p_bronze_container)
arch_container_client = blob_service_client.get_container_client(p_bronze_container)

print(f"p_bronze_container: {p_bronze_container}")
print(f"p_bronze_container_src_path: {p_bronze_container_src_path}")
print(f"processed_folder_path: {processed_folder_path}")
print(f"p_bronze_container_arch: {p_bronze_container_arch}")
print(f"p_original_source_name: {p_original_source_name}")

In [37]:
#function to convert to default date format
def update_filename(file_name, fmt):
    for i in range(len(tokens)):
        for j in range(i+1, min(i+4, len(tokens))+1):
            candidate = " ".join(tokens[i:j])
            if "_" in fmt:
                fmt_clean = fmt.replace("_", " ")
                candidate = candidate.replace("_", " ")
            else:
                fmt_clean = fmt
            try:
                parsed = datetime.strptime(candidate, fmt_clean)
                normalized_date = parsed.strftime("%Y%m%d")
                # New file Name
                prefix = "_".join(tokens[:i])  # everything before the date, for example: 'ATI'
                new_filename = prefix + "_" + normalized_date + ext
                print("Renamed:", new_filename)
                return new_filename
                break
            except ValueError:
                continue

In [38]:
# Iterate  blobs
for blob in src_container_client.list_blobs(name_starts_with=p_bronze_container_src_path):
    blob_name = blob.name
    # print(f"Processing Blob: {blob_name}")

    #Normalize Date structure
    path = os.path.dirname(blob_name)
    file_name = os.path.basename(blob_name)
    base = file_name.split('/')[-1].split('.')[0]
    ext = os.path.splitext(file_name)[1]
    tokens = re.split(r'[_\s]+', base)
    new_file_name = file_name

    if p_date_normalization_enabled is True:

        new_filename = update_filename (file_name,fmt)


        # Define old and new blob names
        current_filename = file_name
        if path:
            new_filename = f"{path}/{new_filename}"
            current_filename = f"{path}/{current_filename}"

        else:
            new_filename = new_file_name
            current_filename = current_filename
        old_file = src_container_client.get_blob_client(current_filename)
        new_file = src_container_client.get_blob_client(new_filename)
        print(current_filename)
        print(new_filename)

        

        # Copy old file to new file with a new name
        copy_source = old_file.url
        new_file.start_copy_from_url(copy_source)
        old_file.delete_blob()


    if blob_name.endswith(('.xlsx', '.xlsm')) and p_original_source_name in blob_name:
        try:
            blob_data = src_container_client.download_blob(new_filename).readall()
            # print(f"p_original_source_name vs blob_name: {blob_name}")
            # print(f"p_original_source_name vs blob_name: {p_original_source_name}")
        except Exception as e:
            print(f"Error downloading file: {blob_name}. Skipping this file. Error: {e}")
            continue

        excel_file = io.BytesIO(blob_data)
        workbook = openpyxl.load_workbook(excel_file, data_only=True)

        # Process worksheets
        worksheets = p_worksheet.split('|')
        for worksheet_name in worksheets:
            if worksheet_name in workbook.sheetnames:
                worksheet = workbook[worksheet_name]
                # print(f"worksheet: {worksheet}")
                data = worksheet.values

                df = pd.DataFrame(data)
                df.columns = df.iloc[0]
                df = df[1:]

                original_file_name = os.path.splitext(new_filename.split('/')[-1])[0]
                # print(f"original_file_name: {original_file_name}")
                main_identifier = original_file_name.split('_')[0]
                # print(f"main_identifier: {main_identifier}")
                remaining_name = original_file_name[len(main_identifier):]
                # print(f"remaining_name: {remaining_name}")

                output_filename = f"{main_identifier} {worksheet_name}{remaining_name}.xlsx"
                print(f"Generated Output Filename: {output_filename}")
                processed_blob_path = f"{processed_folder_path}/{output_filename}"
                # print(f"processed_blob_path: {processed_blob_path}")

                output_stream = io.BytesIO()
                with pd.ExcelWriter(output_stream, engine="openpyxl") as writer:
                    df.to_excel(writer, index=False)
                output_stream.seek(0)

                try:
                    des_container_client.upload_blob(name=processed_blob_path, data=output_stream.getvalue(), overwrite=True)
                except Exception as e:
                    print(f"Error uploading processed file: {processed_blob_path}. Error: {e}")

        # Move to archive
        try:
            archive_blob_path = f"{p_bronze_container_arch}/{new_filename.split('/')[-1]}"
            print(f"Moving Original Blob to Archive: {archive_blob_path}")
            arch_container_client.upload_blob(name=archive_blob_path, data=blob_data, overwrite=True)
            src_container_client.delete_blob(new_filename)
        except Exception as e:
            print(f"Error archiving file: {new_filename}. Error: {e}")

for processed_blob in des_container_client.list_blobs(name_starts_with=processed_folder_path):
    try:
        processed_blob_data = des_container_client.download_blob(processed_blob.name).readall()
        new_path = f"{p_bronze_container_src_path}/{processed_blob.name.split('/')[-1]}"
        print(f"Uploading Processed file to Source Path: {new_path}")
        src_container_client.upload_blob(name=new_path, data=processed_blob_data, overwrite=True)
        des_container_client.delete_blob(processed_blob.name)
    except Exception as e:
        print(f"Error moving processed file: {processed_blob.name}. Error: {e}")

print(f"Delete files in Processed Folder: {processed_folder_path}")
for processed_blob in des_container_client.list_blobs(name_starts_with=processed_folder_path):
    try:
        des_container_client.delete_blob(processed_blob.name)
    except Exception as e:
        print(f"Error deleting file in processed folder: {processed_blob.name}. Error: {e}")
# print("Processed Folder Cleanup Complete!")